# FOLIO: 49 Verified-But-Wrong Cases - Pattern Analysis

Complete manual analysis of all 49 cases where Lean verification succeeds but produces wrong answer.

**Method**: Examined actual Lean code for representative cases from each pattern to identify specific errors.

In [ ]:
import json
import pandas as pd

# Load FOLIO Lean results
with open('../results/folio/lean/all_results.json', 'r') as f:
    lean_results = json.load(f)

# Get all 49 verified but wrong cases
verified_wrong = []
for r in lean_results:
    if r.get('lean_verification', {}).get('success', False) and not r['correct']:
        verified_wrong.append(r)

print(f"Total: {len(verified_wrong)} cases")

# Group by pattern
patterns = {}
for case in verified_wrong:
    pattern = f"{case.get('ground_truth')} → {case.get('prediction')}"
    patterns[pattern] = patterns.get(pattern, 0) + 1

print("\nPattern Distribution:")
for pattern, count in sorted(patterns.items(), key=lambda x: -x[1]):
    print(f"  {pattern:<25} {count:>2} cases ({count/49*100:>5.1f}%)")

## Pattern 1: False → True (21 cases, 42.9%)

**The Problem**: Model axiomatizes conclusions that directly contradict the premises.

### Example 1.1: ID 325 - Nazi Reichstag

**Premises**:
- Heinrich Schmidt was a German politician
- Heinrich Schmidt was a member of the Nazi Reichstag

**Conclusion**: "No politicians are part of the Nazi Reichstag"

**Ground Truth**: False (Heinrich is both!)

**Model Prediction**: True

**Problem in Lean Code**:
```lean
axiom no_politicians_in_nazi : ∀ x, Politician x → ¬ MemberNaziReichstag x
```

**Analysis**: Model axiomatized the conclusion directly, which contradicts the premise that Heinrich is both a politician AND a Nazi Reichstag member. The model even writes `theorem contradiction : False` acknowledging the contradiction, but still predicts True!

In [ ]:
# Show Example 325 code
ex325 = next(r for r in verified_wrong if r.get('example_id') == 325)
print("Example 325 - Lean Code:")
print("="*70)
code = ex325.get('lean_code', '')
for i, line in enumerate(code.split('\n'), 1):
    if i <= 20 or 'no_politicians' in line or 'contradiction' in line:
        print(f"{i:3d}: {line}")

### Example 1.2: ID 304 - Brazilian Footballer

**Premises**:
- Ailton Silva is Brazilian
- Ailton Silva plays for Nautico

**Conclusion**: "No one playing for Nautico is Brazilian"

**Ground Truth**: False

**Model Prediction**: True

**Problem**: Model has axioms for both facts (Ailton is Brazilian, Ailton plays for Nautico) but still predicts the negative conclusion is True. It likely axiomatized a rule that ignores the specific individual.

In [ ]:
# Show Example 304 key lines
ex304 = next(r for r in verified_wrong if r.get('example_id') == 304)
print("Example 304 - Key Lines:")
print("="*70)
code = ex304.get('lean_code', '')
for i, line in enumerate(code.split('\n'), 1):
    if 'brazilian' in line.lower() or 'nautico' in line.lower():
        print(f"{i:3d}: {line}")

### Example 1.3: ID 703 - Humans Are Horses

**Conclusion**: "Some humans are horses"

**Ground Truth**: False (obviously!)

**Model Prediction**: True

**Analysis**: Model axiomatized an absurd universal statement.

In [ ]:
ex703 = next(r for r in verified_wrong if r.get('example_id') == 703)
print(f"Example 703:")
print(f"Premises: {ex703.get('premises')[:200]}...")
print(f"Conclusion: {ex703.get('conclusion')}")
print(f"Truth: {ex703.get('ground_truth')} | Prediction: {ex703.get('prediction')}")

## Pattern 2: Uncertain → True (18 cases, 36.7%)

**The Problem**: Model axiomatizes conclusions about entities or topics NOT mentioned in premises.

### Example 2.1: ID 2 - Joey the Turkey

**Premises**: All about TOM (types of wild turkey, Tom is a wild turkey, etc.)

**Conclusion**: "Joey is a wild turkey"

**Ground Truth**: Uncertain (Joey never mentioned!)

**Model Prediction**: True

**Problem in Lean Code**:
```lean
axiom Joey : Entity
axiom Joey_is_Wild : WildTurkey Joey  <-- PROBLEM!
```

**Analysis**: Model declares Joey exists, then immediately axiomatizes Joey_is_Wild. But Joey is never mentioned in premises! Should be Uncertain.

In [ ]:
ex2 = next(r for r in verified_wrong if r.get('example_id') == 2)
print("Example 2 - Joey:")
print("="*70)
print(f"Premises:\n{ex2.get('premises')}")
print(f"\nConclusion: {ex2.get('conclusion')}")
print(f"Truth: {ex2.get('ground_truth')} | Prediction: {ex2.get('prediction')}")
print("\nKey lines with 'Joey':")
code = ex2.get('lean_code', '')
for i, line in enumerate(code.split('\n'), 1):
    if 'Joey' in line:
        marker = '  <-- PROBLEM!' if 'Wild' in line and 'axiom' in line.lower() else ''
        print(f"{i:3d}: {line}{marker}")

### Example 2.2: ID 564 - Flu

**Premises**: About Monkeypox (no mention of flu)

**Conclusion**: "No one gets the flu"

**Ground Truth**: Uncertain

**Model Prediction**: True

**Problem**:
```lean
axiom no_flu : ∀ x : Being, ¬ GetsFlu x
```

**Analysis**: Flu is not mentioned in premises. Model axiomatized the conclusion about an unrelated topic.

In [ ]:
ex564 = next(r for r in verified_wrong if r.get('example_id') == 564)
print("Example 564 - Flu:")
print("="*70)
print(f"Premises (first 150 chars): {ex564.get('premises')[:150]}...")
print(f"\nConclusion: {ex564.get('conclusion')}")
print(f"Truth: {ex564.get('ground_truth')} | Prediction: {ex564.get('prediction')}")
print("\nLines with 'flu':")
code = ex564.get('lean_code', '')
for i, line in enumerate(code.split('\n'), 1):
    if 'flu' in line.lower() and 'axiom' in line.lower():
        marker = '  <-- PROBLEM!' if 'no_flu' in line or '¬ GetsFlu' in line else ''
        print(f"{i:3d}: {line}{marker}")

### Example 2.3: ID 1208 - James's Lunch

**Conclusion**: "James has lunch in the company"

**Ground Truth**: Uncertain

**Model Prediction**: True

**Analysis**: Premises don't mention where James has lunch. Model axiomatized specific fact without evidence.

In [ ]:
ex1208 = next(r for r in verified_wrong if r.get('example_id') == 1208)
print(f"Example 1208:")
print(f"Conclusion: {ex1208.get('conclusion')}")
print(f"Truth: {ex1208.get('ground_truth')} | Prediction: {ex1208.get('prediction')}")

## Pattern 3: False → Unknown (5 cases, 10.2%)

**The Problem**: Model fails to derive contradiction from premises that prove statement is False.

In [ ]:
# Show False → Unknown cases
false_unknown = [r for r in verified_wrong if r.get('ground_truth') == 'False' and r.get('prediction') == 'Unknown']
print(f"False → Unknown cases: {len(false_unknown)}")
print("="*70)
for case in false_unknown[:3]:
    print(f"\nID {case.get('example_id')}: {case.get('conclusion')[:60]}...")
    print(f"  Truth: False | Prediction: Unknown")

## Pattern 4: True → Unknown (4 cases, 8.2%)

**The Problem**: Model fails to derive valid conclusion from premises.

In [ ]:
# Show True → Unknown cases
true_unknown = [r for r in verified_wrong if r.get('ground_truth') == 'True' and r.get('prediction') == 'Unknown']
print(f"True → Unknown cases: {len(true_unknown)}")
print("="*70)
for case in true_unknown:
    print(f"\nID {case.get('example_id')}: {case.get('conclusion')[:60]}...")
    print(f"  Truth: True | Prediction: Unknown")

## Pattern 5: Uncertain → False (1 case, 2.0%)

**The Problem**: Model incorrectly proves negative for uncertain statement.

In [ ]:
ex658 = next(r for r in verified_wrong if r.get('example_id') == 658)
print("Example 658 - Beijing:")
print("="*70)
print(f"Conclusion: {ex658.get('conclusion')}")
print(f"Truth: {ex658.get('ground_truth')} | Prediction: {ex658.get('prediction')}")
print("\nKey line:")
code = ex658.get('lean_code', '')
for i, line in enumerate(code.split('\n'), 1):
    if 'beijing_north' in line.lower():
        print(f"{i:3d}: {line}  <-- Axiomatizes Beijing is in NORTH, proves conclusion False")

## Summary: Key Findings

In [ ]:
print("="*70)
print("SUMMARY: ERROR PATTERNS IN 49 VERIFIED-BUT-WRONG CASES")
print("="*70)

print("\n1. False → True (21 cases, 42.9%)")
print("   Problem: Axiomatizes conclusions that contradict premises")
print("   Example: Premises say 'X is Y', conclusion 'No X is Y', model says True")

print("\n2. Uncertain → True (18 cases, 36.7%)")
print("   Problem: Axiomatizes facts about entities/topics NOT in premises")
print("   Example: Premises about Tom, conclusion about Joey, model says True")

print("\n3. False/True → Unknown (9 cases, 18.4%)")
print("   Problem: Fails to derive valid conclusions or contradictions")
print("   Example: Model gives up, says Unknown when answer is derivable")

print("\n4. Uncertain → False (1 case, 2.0%)")
print("   Problem: Incorrectly proves negative")

print("\n" + "="*70)
print("KEY INSIGHT:")
print("="*70)
print("39/49 cases (79.6%) are Patterns 1 & 2: Axiomatizing conclusions")
print("")
print("The dominant error is NOT reasoning failure - it's that models")
print("assume conclusions as axioms instead of proving them from premises.")
print("")
print("Lean verification accepts these because they're syntactically valid,")
print("but they're logically incorrect: the 'proof' assumes what it should prove.")
print("="*70)